In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

---
# MANUAL-NEUTRALIZE

In [ ]:
def count_threshold(seq: list, limit: float) -> int:
    count = 0
    for num in seq:
        if num > limit:
            count += 1
    return count

In [ ]:
sfGFP_tx = pd.read_csv("TX_sfGFP_1_5.csv")
sfGFP_tx_ini = pd.read_csv("TX_sfGFP_Shiv.csv")

In [ ]:
sfGFP_tx_pos = sfGFP_tx[sfGFP_tx['strand  [Strand Orientation]'] == "+"]
sfGFP_tx_ini_pos = sfGFP_tx_ini[sfGFP_tx_ini['strand  [Strand Orientation]'] == "+"]

sfGFP_tx_rv = sfGFP_tx[sfGFP_tx['strand  [Strand Orientation]'] == "-"]
sfGFP_tx_ini_rv = sfGFP_tx_ini[sfGFP_tx_ini['strand  [Strand Orientation]'] == "-"]

In [ ]:
tx_list_pos = sfGFP_tx_pos["Tx_rate  [Transcription Initiation Rate (au)]"]
tx_list_ini_pos = sfGFP_tx_ini_pos["Tx_rate  [Transcription Initiation Rate (au)]"]

tx_list_rv = sfGFP_tx_rv["Tx_rate  [Transcription Initiation Rate (au)]"]
tx_list_ini_rv = sfGFP_tx_ini_rv["Tx_rate  [Transcription Initiation Rate (au)]"]

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(10,4), sharey = True)

axs[0, 0].plot(sfGFP_tx_ini_pos["TSS  [Transcriptional Start Site Position (nt)]"],
        tx_list_ini_pos,
        color='g')
axs[0, 0].set_ylabel("FW")
axs[0, 0].set_title("Original")

axs[0, 1].plot(sfGFP_tx_pos["TSS  [Transcriptional Start Site Position (nt)]"],
        tx_list_pos,
        color='g')
axs[0, 1].set_title("Polished")

axs[1, 0].plot(sfGFP_tx_ini_rv["TSS  [Transcriptional Start Site Position (nt)]"],
        tx_list_ini_rv,
        color='g')
axs[1, 0].set_ylabel("RV")
axs[1, 1].plot(sfGFP_tx_rv["TSS  [Transcriptional Start Site Position (nt)]"],
        tx_list_rv,
        color='g')

fig.text(0.02, 0.5, 'Predicted Tx Rate (a.u.) ', va='center', rotation='vertical')
fig.text(0.5, 0.00, 'Position', ha='center')

plt.show()

---
# AUTO-NEUTRALIZE

In [ ]:
from Promoter_Calculator import *

In [ ]:
hex35_rank = {"first":["TTG", "TTT", "ATG", "TAG",
                  "TGG", "TTC", "CTG", "GTG",
                  "GTA", "TCT", "CTA", "TAT",
                  "TTA", "TCG", "TGA", "CTT",
                  "TAA", "GTC", "GTT", "TGT",
                  "ATC", "GCG", "ATT", "CGT",
                  "AAA", "AGC", "GGA", "GAT",
                  "AGA", "AGT", "ACT", "ACC",
                  "TGC", "AAT", "CCG", "CGG",
                  "TAC", "TCC", "CAG", "GAC",
                  "TCA", "GGT", "CTC", "ACG",
                  "AGG", "GCT", "CAT", "CCT",
                  "CAC", "AAC", "CGA", "GAG",
                  "CCA", "GGC", "AAG", "CAA",
                  "ATA", "GGG", "GCC", "GAA",
                  "ACA", "CCC", "CGC", "GCA"],
        "second":["ACA", "ACT", "CCT", "AAT",
                  "CCA", "CAA", "ATC", "AAA",
                  "GTA", "TAA", "ACG", "ATT",
                  "AGA", "CTA", "TGT", "TTT",
                  "GAA", "TCT", "TCA", "CTT",
                  "CAT", "TAT", "ATA", "CAG",
                  "GCA", "GCT", "CCG", "ACC",
                  "TAG", "CAC", "CGT", "GCG",
                  "TGG", "TTG", "AAG", "CTG",
                  "CGG", "GAT", "GAG", "ATG",
                  "AAC", "CGA", "CCC", "TTA",
                  "TGA", "AGG", "GTT", "GTG",
                  "TCG", "AGT", "GGA", "TGC",
                  "CTC", "CGC", "TAC", "AGC",
                  "TTC", "GCC", "GGG", "GTC",
                  "GAC", "TCC", "GGC", "GGT"]}

hex10_rank = {"first":["TAT", "TAA", "TAC", "TAG",
                  "AAT", "CAT", "GAT", "GTA",
                   "CAA", "TTA", "CTA", "CAC",
                  "ATT", "GAC", "GGG", "CTG",
                  "GGC", "CGG", "CGC", "GTC",
                  "GGA", "GTG", "CTC", "GCC",
                  "ATG", "ATC", "ATA", "AGG",
                  "AGC", "AGA", "ACG", "ACC",
                  "ACA", "CCG", "GCG", "GGT",
                  "TGC", "TCC", "AAC", "TGT",
                  "TTG", "TGG", "CGA", "TCT",
                  "TGA", "CAG", "AAA", "GAA",
                  "AAG", "GAG", "ACT", "CCA",
                  "GCA", "TCA", "AGT", "CCC",
                  "GCT", "GTT", "TTC", "TCG",
                  "TTT", "CTT", "CGT", "CCT"],
         "second":["AAT", "TAT", "ACT", "AGT",
                   "CAT", "ATT", "GAT", "TCT",
                   "GCT", "TGT", "CCT", "ATA",
                   "GGT", "TTT", "GTT", "CGT",
                   "CTT", "ACA", "AAA", "AGG",
                   "TTC", "AAC", "ATG", "CTG",
                   "CGG", "GGG", "CGA", "TGG",
                   "TAG", "ATC", "ACC", "TAA",
                   "AAG", "CTA", "TTA", "AGA",
                   "TGA", "GTA", "CTC", "AGC",
                   "GAA", "ACG", "TAC", "CAC",
                   "GAC", "GCC", "CCG", "GCG",
                   "GCA", "TCC", "GTC", "CCC",
                   "CAG", "TCG", "TTG", "GGA",
                   "TCA", "GAG", "GTG", "GGC",
                   "CAA", "CCA", "TGC", "CGC",]}
codon_usage = {"codon":['TTT', 'TTC', 'TTA', 'TTG', 'TAT', 'TAC', 'TAA', 'TAG', 'CTT', 'CTC', 'CTA', 'CTG', 'CAT', 'CAC', 'CAA', 'CAG', 'ATT', 'ATC', 'ATA', 'ATG', 'AAT', 'AAC', 'AAA', 'AAG', 'GTT', 'GTC', 'GTA', 'GTG', 'GAT', 'GAC', 'GAA', 'GAG', 'TCT', 'TCC', 'TCA', 'TCG', 'TGT', 'TGC', 'TGA', 'TGG', 'CCT', 'CCC', 'CCA', 'CCG', 'CGT', 'CGC', 'CGA', 'CGG', 'ACT', 'ACC', 'ACA', 'ACG', 'AGT', 'AGC', 'AGA', 'AGG', 'GCT', 'GCC', 'GCA', 'GCG', 'GGT', 'GGC', 'GGA', 'GGG'],
               "aa":['F', 'F', 'L', 'L', 'Y', 'Y', '*', '*', 'L', 'L', 'L', 'L', 'H', 'H', 'Q', 'Q', 'I', 'I', 'I', 'M', 'N', 'N', 'K', 'K', 'V', 'V', 'V', 'V', 'D', 'D', 'E', 'E', 'S', 'S', 'S', 'S', 'C', 'C', '*', 'W', 'P', 'P', 'P', 'P', 'R', 'R', 'R', 'R', 'T', 'T', 'T', 'T', 'S', 'S', 'R', 'R', 'A', 'A', 'A', 'A', 'G', 'G', 'G', 'G'],
               "freq":[0.58, 0.42, 0.14, 0.13, 0.59, 0.41, 0.61, 0.09, 0.12, 0.10, 0.04, 0.47, 0.57, 0.43, 0.34, 0.66, 0.49, 0.39, 0.11, 1.00, 0.49, 0.51, 0.74, 0.26, 0.28, 0.20, 0.17, 0.35, 0.63, 0.37, 0.68, 0.32, 0.17, 0.15, 0.14, 0.14, 0.46, 0.54, 0.30, 1.00, 0.18, 0.13, 0.20, 0.49, 0.36, 0.36, 0.07, 0.11, 0.19, 0.40, 0.17, 0.25, 0.16, 0.25, 0.07, 0.04, 0.18, 0.26, 0.23, 0.33, 0.35, 0.37, 0.13, 0.15]}

In [ ]:
CDSin = ""
CDSin = CDSin.upper()

CDS_tx_ini = pd.read_csv("TX_sfGFP_auto2000_pass2.csv")

tx_threshold = 2000
freq_min = 0.1

In [ ]:
spikes = CDS_tx_ini[CDS_tx_ini['Tx_rate  [Transcription Initiation Rate (au)]'] > tx_threshold]
cores = []
for idx in spikes.index:
    cores.append(spikes.loc[idx, 'strand  [Strand Orientation]']+spikes.loc[idx, 'hex35 [-35 hexamer sequence]'] + spikes.loc[idx, 'spacer [spacer sequence]'] + spikes.loc[idx, 'hex10 [-10 hexamer sequence]'])
cores = set(cores)

In [ ]:
def revcomp_DNA(dna):
    dna = dna.upper()
    dna = dna.replace("A", "t").replace("T", "a").replace("C", "g").replace("G", "c").upper()
    return dna[::-1]

def translate_CDS(prot):
    trans = ""
    for pos in range(0, len(prot), 3):
        codon = prot[pos:pos+3]
        aa = codon_usage["aa"][codon_usage["codon"].index(codon)]
        trans += aa
    return trans

def combine(list1, list2, sep=""):
    new = []
    for el in list1:
        for el2 in list2:
            new.append(el+sep+el2)
    return(new)

def get_alt_codons(query, freq_min = 0.1):
    aa_idx = [idx for idx, aa in enumerate (codon_usage["aa"]) if aa == query]
    return [codon_usage["codon"][idx] for idx in aa_idx if codon_usage["freq"][idx] >= freq_min]

def neutralize_hexamer(bigCDS, pos: int, mode="10", strand = "+", freq_min = 0.1):
    hexamer = bigCDS[pos:pos+6]
    match strand:
        case "+":
            aff_codons = bigCDS[pos//3 * 3:pos//3 * 3 + 6]
        case "-":
            revpos = len(bigCDS) - pos
            aff_codons = revcomp_DNA(bigCDS)[revpos//3 * 3 - 6:revpos//3 * 3]
            
    aa = translate_CDS(aff_codons)
    alt_codons = combine(get_alt_codons(aa[0], freq_min), get_alt_codons(aa[1], freq_min))
    match strand:
        case "+":
            alt_hexamer = [(alt + (bigCDS[pos // 3 * 3:pos + 6][6:]))[-6:] for alt in alt_codons]
        case "-":
            alt_hexamer = [((bigCDS[pos:len(bigCDS) - (revpos//3 * 3) + 6][:-6]) + revcomp_DNA(alt))[:6] for alt in alt_codons]
    firsts = list(set([opt[:3] for opt in alt_hexamer]))
    secs = list(set([opt[-3:] for opt in alt_hexamer]))
    match mode:
        case "10":
            first_ranks = [hex10_rank["first"].index(triplet) for triplet in firsts]
            sec_ranks = [hex10_rank["second"].index(triplet) for triplet in secs]
        case "35":
            first_ranks = [hex35_rank["first"].index(triplet) for triplet in firsts]
            sec_ranks = [hex35_rank["second"].index(triplet) for triplet in secs]
        case _:
            return hexamer
    if first_ranks == [] or sec_ranks == []:
        return hexamer
    best = firsts[first_ranks.index(max(first_ranks))] + secs[sec_ranks.index(max(sec_ranks))]
    return best

In [ ]:
newCDS = CDSin
for core in cores:
    orientation = core[0]
    match orientation:
        case "+":
            pass
            pos = CDSin.find(core[1:])
            new35 = neutralize_hexamer(CDSin, pos, "35", "+")
            new10 = neutralize_hexamer(CDSin, pos+len(core[1:])-6, "10", "+")
            newCDS = newCDS[:pos] + new35 + newCDS[pos+6:]
            newCDS = newCDS[:pos+len(core[1:])-6] + new10 + newCDS[pos+len(core[1:]):]
            print("replaced", core, "for", newCDS[pos:pos+len(core[1:])], "at", pos)

        case "-":
            pass
            revCDS = revcomp_DNA(CDSin)
            pos = revCDS.find(core[1:])
            new35 = neutralize_hexamer(revCDS, pos, "35", "-")
            new10 = neutralize_hexamer(revCDS, pos+len(core[1:])-6, "10", "-")
            revpos = len(CDSin) - pos
            newCDS = newCDS[:revpos-6] + revcomp_DNA(new35) + newCDS[revpos:]
            newCDS = newCDS[:revpos-len(core[1:])] + revcomp_DNA(new10) + newCDS[revpos-len(core[1:])+6:]
            print("replaced", "-"+revcomp_DNA(core[1:]), "for", newCDS[revpos-len(core[1:]):revpos], "at", revpos)
    print()
assert len(CDSin) == len(newCDS), "TX_neutralization: length of input and output sequences don't match"
assert translate_CDS(CDSin) == translate_CDS(newCDS), "TX_neutralization: aminoacid sequences of input and output don't match"

---
# INTEGRATING SALIS CALC

In [ ]:
calc = Promoter_Calculator_v1_0.Promoter_Calculator()

In [ ]:
CDSin = ""
CDSin = CDSin.upper()

In [ ]:
calc.run(CDSin, TSS_range=[0, len(CDSin)])
output = calc.output()

In [ ]:
fw_pred = pd.DataFrame.from_dict(output['Forward_Predictions_per_TSS'], orient="index")
start_sites = fw_pred["TSS"]
tx_rates = fw_pred["Tx_rate"]

In [ ]:
plt.figure(figsize = (10, 2))
plt.plot(start_sites, tx_rates)
plt.show()

In [ ]:
rv_pred = pd.DataFrame.from_dict(output['Reverse_Predictions_per_TSS'], orient="index")
start_sites = rv_pred["TSS"][::-1]
tx_rates = rv_pred["Tx_rate"]

In [ ]:
plt.figure(figsize = (10, 2))
plt.plot(start_sites, tx_rates)
plt.show()

In [ ]:
fw_spikes = fw_pred[fw_pred["Tx_rate"] >= 2000]
rv_spikes = rv_pred[rv_pred["Tx_rate"] >= 2000]

In [ ]:
cores = []
for idx in fw_spikes.index:
    cores.append("+"+fw_spikes.loc[idx, 'hex35'] + fw_spikes.loc[idx, 'spacer'] + fw_spikes.loc[idx, 'hex10'])
for idx in rv_spikes.index:
    cores.append("-"+rv_spikes.loc[idx, 'hex35'] + rv_spikes.loc[idx, 'spacer'] + rv_spikes.loc[idx, 'hex10'])
cores = set(cores)

In [ ]:
fw_spikes

In [ ]:
def neutralize_CDS(CDSin, threshold, freq_min = 0.1, max_iter = 3):
    CDSin = CDSin.upper()

    assert len(CDSin) % 3 == 0, "TX_neutralization: CDS length is not a multiple of 3"

    iter_no = 0
    while True:
        iter_no += 1
        print("Iteration", iter_no)
        
        calc.run(CDSin, TSS_range=[0, len(CDSin)])
        output = calc.output()
        fw_pred = pd.DataFrame.from_dict(output['Forward_Predictions_per_TSS'], orient="index")
        rv_pred = pd.DataFrame.from_dict(output['Reverse_Predictions_per_TSS'], orient="index")
    
        fw_spikes = fw_pred[fw_pred["Tx_rate"] >= threshold]
        rv_spikes = rv_pred[rv_pred["Tx_rate"] >= threshold]
    
        cores = []
        for idx in fw_spikes.index:
            cores.append("+"+fw_spikes.loc[idx, 'hex35'] + fw_spikes.loc[idx, 'spacer'] + fw_spikes.loc[idx, 'hex10'])
        for idx in rv_spikes.index:
            cores.append("-"+rv_spikes.loc[idx, 'hex35'] + rv_spikes.loc[idx, 'spacer'] + rv_spikes.loc[idx, 'hex10'])
        cores = set(cores)

        if len(cores) == 0:
            print("Reached no peaks above threshold:", threshold)
            break
        
        newCDS = CDSin
    
        for core in cores:
            orientation = core[0]
            match orientation:
                case "+":
                    pass
                    pos = CDSin.find(core[1:])
                    new35 = neutralize_hexamer(CDSin, pos, "35", "+", freq_min = freq_min)
                    new10 = neutralize_hexamer(CDSin, pos+len(core[1:])-6, "10", "+", freq_min = freq_min)
                    newCDS = newCDS[:pos] + new35 + newCDS[pos+6:]
                    newCDS = newCDS[:pos+len(core[1:])-6] + new10 + newCDS[pos+len(core[1:]):]
                    print("replaced", core, "for", newCDS[pos:pos+len(core[1:])], "at", pos)
        
                case "-":
                    pass
                    revCDS = revcomp_DNA(CDSin)
                    pos = revCDS.find(core[1:])
                    new35 = neutralize_hexamer(revCDS, pos, "35", "-", freq_min=freq_min)
                    new10 = neutralize_hexamer(revCDS, pos+len(core[1:])-6, "10", "-", freq_min=freq_min)
                    revpos = len(CDSin) - pos
                    newCDS = newCDS[:revpos-6] + revcomp_DNA(new35) + newCDS[revpos:]
                    newCDS = newCDS[:revpos-len(core[1:])] + revcomp_DNA(new10) + newCDS[revpos-len(core[1:])+6:]
                    print("replaced", "-"+revcomp_DNA(core[1:]), "for", newCDS[revpos-len(core[1:]):revpos], "at", revpos)
            print()
        assert len(CDSin) == len(newCDS), "TX_neutralization: length of input and output sequences don't match"
        assert translate_CDS(CDSin) == translate_CDS(newCDS), "TX_neutralization: aminoacid sequences of input and output don't match"
        CDSin = newCDS
        if iter_no >= max_iter:
            print("reached max iterations:", iter_no)
            break
    return newCDS

In [ ]:
oldCDS = ""

newCDS = neutralize_CDS(oldCDS, 2000, max_iter=5, freq_min = 0.2)

In [ ]:
calc.run(oldCDS.upper(), TSS_range=[0, len(oldCDS)])
output_old = calc.output()

calc.run(newCDS, TSS_range=[0, len(newCDS)])
output_new = calc.output()

In [ ]:
fw_pred_old = pd.DataFrame.from_dict(output_old['Forward_Predictions_per_TSS'], orient="index")
fw_old_tss = fw_pred_old["TSS"]
fw_old_tx = fw_pred_old["Tx_rate"]

rv_pred_old = pd.DataFrame.from_dict(output_old['Reverse_Predictions_per_TSS'], orient="index")
rv_old_tss = rv_pred_old["TSS"][::-1]
rv_old_tx = rv_pred_old["Tx_rate"]

fw_pred_new = pd.DataFrame.from_dict(output_new['Forward_Predictions_per_TSS'], orient="index")
fw_new_tss = fw_pred_new["TSS"]
fw_new_tx = fw_pred_new["Tx_rate"]

rv_pred_new = pd.DataFrame.from_dict(output_new['Reverse_Predictions_per_TSS'], orient="index")
rv_new_tss = rv_pred_new["TSS"][::-1]
rv_new_tx = rv_pred_new["Tx_rate"]

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(10,4), sharey = True, sharex = True)

axs[0, 0].plot(fw_old_tss, fw_old_tx, color='k')
axs[0, 0].set_ylabel("FW")
axs[0, 0].set_title("Original")
axs[0, 0].grid()
axs[0, 0].set_axisbelow(True)

axs[0, 1].plot(fw_new_tss, fw_new_tx, color='k')
axs[0, 1].set_title("Polished")
axs[0, 1].grid()
axs[0, 1].set_axisbelow(True)

axs[1, 0].plot(rv_old_tss, rv_old_tx, color='r')
axs[1, 0].set_ylabel("RV")
axs[1, 0].grid()
axs[1, 0].set_axisbelow(True)

axs[1, 1].plot(rv_new_tss, rv_new_tx, color='r')
axs[1, 1].grid()
axs[1, 1].set_axisbelow(True)


fig.text(0.02, 0.5, 'Predicted Tx Rate (a.u.) ', va='center', rotation='vertical')
fig.text(0.5, 0.00, 'Position', ha='center')

plt.show()